In [ ]:
import os
import pandas as pd
import numpy as np
print(os.getcwd())

In [ ]:
df = pd.read_csv('labels_brset.csv')

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
'patient_id', 'DR_ICDR'
def StratifiedGroupShuffleSplit(df, y_col, group_col, test_size=0.3, shuffle = False, random_state=42):
    groups = df.groupby(y_col)
    all_train = []
    all_test = []
    for group_id, group in groups:
        # if a group is already taken in test or train it must stay there
        group = group[~group[group_col].isin(all_train+all_test)]
        # if group is empty 
        if group.shape[0] == 0:
            continue
        train_inds, test_inds = next(GroupShuffleSplit(
            test_size=test_size, n_splits=2, random_state=random_state).split(group, groups=group[group_col]))

        all_train += group.iloc[train_inds][group_col].tolist()
        all_test += group.iloc[test_inds][group_col].tolist()
    
    train= df[df[group_col].isin(all_train)]
    test= df[df[group_col].isin(all_test)]

    form_train = set(train[group_col].tolist())
    form_test = set(test[group_col].tolist())
    inter = form_train.intersection(form_test)  
    print(f'Number of overlaping patients: {inter}')
    print(f'Number of patients in train and test: {len(train)}({len(train)/len(df)}), {len(test)}({len(test)/len(df)})')
    print(f'Label distribution in train: {train[y_col].value_counts()/len(train)}')
    print(f'Label distribution in test: {test[y_col].value_counts()/len(test)}')
    return train, test

train, test = StratifiedGroupShuffleSplit(df, 'DR_ICDR', 'patient_id', test_size=0.3, random_state=42)
test, val = StratifiedGroupShuffleSplit(test, 'DR_ICDR', 'patient_id', test_size=0.2, random_state=42)
train.to_csv('train_brset_nooverlap_v1', index=False)
test.to_csv('test_brset_nooverlap_v1', index=False)
val.to_csv('val_brset_nooverlap_v1', index=False)



In [ ]:
df_train = pd.read_csv('data/train_brset.csv')
df_test = pd.read_csv('data/test_brset.csv')
df_val = pd.read_csv('data/val_brset.csv')
df_train_nooverlap = pd.read_csv('data/train_brset_nooverlap.csv')
df_test_nooverlap = pd.read_csv('data/test_brset_nooverlap.csv')
df_val_nooverlap = pd.read_csv('data/val_brset_nooverlap.csv')

In [ ]:
# Find the overlaps of patient IDs between the dataframes
def find_overlap(df_train, df_test, df_val):
    
    overlap_train_test = set(df_train['patient_id']).intersection(set(df_test['patient_id']))
    overlap_train_val = set(df_train['patient_id']).intersection(set(df_val['patient_id']))
    overlap_test_val = set(df_test['patient_id']).intersection(set(df_val['patient_id']))

    print(f"Overlap between train and test: {len(overlap_train_test)}")
    print(f"Overlap between train and val: {len(overlap_train_val)}")
    print(f"Overlap between test and val: {len(overlap_test_val)}")

In [ ]:
find_overlap(df_train, df_test, df_val)
find_overlap(df_train_nooverlap, df_test_nooverlap, df_val_nooverlap)